# 🔎 Georgia RAG — pipeline step by step

This notebook lets you walk through the whole pipeline by hand and inspect what comes out at each stage:

1. Check `.env` and settings
2. (optional) Fetch chat history
3. Raw messages
4. Chunking (you can tweak parameters)
5. Indexing
6. Retrieval
7. RAG answer
8. Debug: which prompt is actually sent to GPT

> Run: `uv run jupyter lab` from the project root, then open `notebooks/explore.ipynb`.

> Note: query strings stay in Russian on purpose — they must match the Russian content of the chats.

In [ ]:
# So that `import config` / `src.*` work from the notebooks/ folder, move to the project root
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

# Autoreload: edits in src/*.py are picked up without restarting the kernel
%load_ext autoreload
%autoreload 2

## 1. Check `.env` and settings

Make sure the keys were picked up (values are masked).

In [ ]:
import importlib, config
importlib.reload(config)

def mask(v):
    return (v[:4] + "…" + str(len(v)) + " chars") if v else "❌ not set"

print("OPENAI_API_KEY  :", mask(config.OPENAI_API_KEY))
print("BOT_TOKEN       :", mask(config.BOT_TOKEN))
print("TELEGRAM_API_ID :", config.TELEGRAM_API_ID or "❌ not set")
print("TELEGRAM_PHONE  :", config.TELEGRAM_PHONE or "❌ not set")
print()
print("Chats:", [c["username"] for c in config.CHATS])
print("Models:", config.EMBED_MODEL, "|", config.CHAT_MODEL)
print("Chunking: gap =", config.CHUNK_TIME_GAP_MIN, "min, max =", config.CHUNK_MAX_CHARS, "chars")

## 2. (optional) Fetch chat history

If you already ran `uv run python -m src.ingest` in the terminal — skip this step.

The first run will ask for the confirmation code from Telegram (entered right in the notebook). After authorization a session file is created, so the code won't be needed again.

In [ ]:
# Uncomment to fetch history directly from the notebook:
#
# from telethon import TelegramClient
# from src.ingest import ingest_chat
#
# client = TelegramClient("georgia_ingest", int(config.TELEGRAM_API_ID), config.TELEGRAM_API_HASH)
# await client.start(phone=config.TELEGRAM_PHONE or None)
# for chat in config.CHATS:
#     await ingest_chat(client, chat)
# await client.disconnect()

## 3. Raw messages

Look at what was fetched: how many messages and what they look like.

In [ ]:
from src.preprocess import _load_raw

username = config.CHATS[0]["username"]
raw_path = config.RAW_DIR / f"{username}.jsonl"
print("File:", raw_path, "| exists:", raw_path.exists())

if raw_path.exists():
    msgs = _load_raw(raw_path)
    print("Total messages:", len(msgs))
    print("\nLast 5:")
    for m in msgs[-5:]:
        sender = m.get("sender") or "Anonymous"
        print(f"  [{m['date'][:16]}] {sender}: {m['text'][:90]}")
else:
    print("Fetch the history first (step 2 or `uv run python -m src.ingest`).")

## 4. Chunking — how messages are merged into dialogs

Build chunks and inspect the result. The `CHUNK_TIME_GAP_MIN` and `CHUNK_MAX_CHARS` parameters can be changed right here to compare.

In [ ]:
from src.preprocess import chunk_messages

# Feel free to experiment with the parameters:
# config.CHUNK_TIME_GAP_MIN = 15
# config.CHUNK_MAX_CHARS = 2000

chunks = chunk_messages(msgs)
sizes = [len(c["text"]) for c in chunks]
print(f"{len(msgs)} messages -> {len(chunks)} chunks")
if sizes:
    print(f"Chunk size (chars): min {min(sizes)}, avg {sum(sizes)//len(sizes)}, max {max(sizes)}")

In [ ]:
# Inspect 3 random chunks in full
import random
for c in random.sample(chunks, min(3, len(chunks))):
    print("=" * 70)
    print(c["link"], "| messages", c["first_msg_id"], "-", c["last_msg_id"])
    print(c["text"])

## 5. Indexing (embeddings → Chroma)

⚠️ This step spends OpenAI tokens (embeddings are cheap but not free). Run it once the chunks look good.

In [ ]:
# Indexing is usually more convenient to run as scripts (they read data/chunks/*.jsonl):
#   uv run python -m src.preprocess   # save chunks to disk
#   uv run python -m src.index        # build the index
#
# Or right here:
from src.index import main as build_index
from src.preprocess import main as build_chunks

build_chunks()   # data/chunks/*.jsonl
build_index()    # embeddings -> chroma_db/

In [ ]:
from src.store import get_collection
col = get_collection()
print("Records in collection:", col.count())

## 6. Retrieval

See which fragments are found for a question and how relevant they are (score closer to 1 = better).

In [ ]:
from src.retrieve import search

query = "как открыть ип в грузии"  # <- change the question (keep it in Russian)

for i, h in enumerate(search(query, k=5), 1):
    print(f"--- #{i}  score={h['score']:.3f}  {h['meta']['link']}")
    print(h["text"][:300])
    print()

## 7. RAG answer

The final answer from GPT based on the retrieved fragments + the list of sources.

In [ ]:
from src.rag import answer

res = answer("какие документы нужны для открытия ип")  # <- change the question

print(res["answer"])
print("\n——— Sources ———")
for s in res["sources"]:
    print(s["title"], "|", s["link"])

## 8. Debug: which prompt is actually sent to GPT

Useful to understand why the model answered the way it did, and to tweak the system prompt in `src/rag.py`.

In [ ]:
from src.rag import _build_context, SYSTEM_PROMPT
from src.retrieve import search

q = "как получить внж"
hits = search(q, k=4)

print("### SYSTEM PROMPT ###\n")
print(SYSTEM_PROMPT)
print("\n### CONTEXT (fragments) ###\n")
print(_build_context(hits))